# Temporary test notebook for pairs_trading/utils.py and pairs_trading/pairs_filter.py
Scratch file — safe to delete once satisfied.

In [2]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))

from pairs_trading.utils import find_pair_groups
from pairs_trading.pairs_filter import find_pairs, compare_pair, load_metadata

## find_pair_groups() — sector/industry bucketing

In [3]:
groups = find_pair_groups()

print('sectors found:', list(groups.keys()))
print()
print('Technology industries:', list(groups['Technology']['industries'].keys()))
print()
print('Semiconductors group:', groups['Technology']['industries']['Semiconductors'])
print()
print('Technology sector ETFs:', groups['Technology']['etfs'])

sectors found: ['Technology', 'Communication Services', 'Consumer Cyclical', 'Consumer Defensive', 'Healthcare', 'Financial Services', 'Energy', 'Industrials', 'Basic Materials', 'Utilities', 'Real Estate']

Technology industries: ['Semiconductors', 'Consumer Electronics', 'Software - Infrastructure', 'Communication Equipment', 'Semiconductor Equipment & Materials', 'Information Technology Services', 'Computer Hardware', 'Software - Application', 'Electronic Components', 'Scientific & Technical Instruments', 'Solar', 'Electronics & Computer Distribution']

Semiconductors group: ['NVDA', 'AVGO', 'MU', 'AMD', 'INTC', 'TXN', 'QCOM', 'ADI', 'MRVL', 'NXPI', 'MPWR', 'MCHP', 'ON', 'GFS', 'ALAB', 'CRDO', 'TSEM', 'MTSI', 'SITM', 'LSCC', 'RMBS', 'SMTC', 'SWKS', 'QRVO', 'CRUS', 'MXL', 'ALGM', 'SLAB', 'VSH', 'SYNA', 'DIOD', 'PI', 'NVTS', 'LASR', 'POWI', 'WOLF', 'POET', 'SKYT', 'AIP', 'AMBQ']

Technology sector ETFs: ['VGT', 'XLK', 'SMH', 'SOXX', 'IYW', 'FTEC', 'BAI', 'IGV']


In [8]:
print(groups['Technology']['industries'])

{'Semiconductors': ['NVDA', 'AVGO', 'MU', 'AMD', 'INTC', 'TXN', 'QCOM', 'ADI', 'MRVL', 'NXPI', 'MPWR', 'MCHP', 'ON', 'GFS', 'ALAB', 'CRDO', 'TSEM', 'MTSI', 'SITM', 'LSCC', 'RMBS', 'SMTC', 'SWKS', 'QRVO', 'CRUS', 'MXL', 'ALGM', 'SLAB', 'VSH', 'SYNA', 'DIOD', 'PI', 'NVTS', 'LASR', 'POWI', 'WOLF', 'POET', 'SKYT', 'AIP', 'AMBQ'], 'Consumer Electronics': ['AAPL', 'SONO'], 'Software - Infrastructure': ['MSFT', 'ORCL', 'PLTR', 'PANW', 'CRWD', 'SNPS', 'FTNT', 'NET', 'CRWV', 'XYZ', 'TWLO', 'ZS', 'VRSN', 'MDB', 'NTAP', 'CPAY', 'FFIV', 'AKAM', 'IOT', 'OKTA', 'GEN', 'DOCN', 'TOST', 'RBRK', 'CHKP', 'NTNX', 'GDDY', 'SAIL', 'CORZ', 'DOX', 'DBX', 'KLAR', 'S', 'PATH', 'BLSH', 'WEX', 'ZETA', 'NTSK', 'ACIW', 'RELY', 'GTLB', 'BOX', 'QLYS', 'DLO', 'CLBT', 'FOUR', 'VRNS', 'PAY', 'TDC', 'WIX', 'NN', 'NTCT', 'TENB', 'EEFT', 'CALX', 'STNE', 'PAGS', 'RAMP', 'INFQ', 'AVPT', 'ATEN', 'FLYW', 'BAND', 'MQ', 'FIVN', 'PAYO', 'APPN', 'EVTC', 'KDK', 'PICS', 'GCT', 'PRGS', 'AI'], 'Communication Equipment': ['CSCO', 'CIEN

In [3]:
# Sanity checks
assert 'Technology' in groups
assert 'Semiconductors' in groups['Technology']['industries']
assert 'NVDA' in groups['Technology']['industries']['Semiconductors']
assert 'XLK' in groups['Technology']['etfs']

all_tickers = {t for s in groups.values() for t in s['etfs']} | \
              {t for s in groups.values() for inds in s['industries'].values() for t in inds}
assert not any(t.endswith('USDT') for t in all_tickers), 'crypto leaked into groups'

print('find_pair_groups() checks passed —', len(all_tickers), 'tickers grouped across', len(groups), 'sectors')

find_pair_groups() checks passed — 2016 tickers grouped across 11 sectors


## pairs_filter — compare_pair() and find_pairs()

In [4]:
meta = load_metadata()

result = compare_pair('NVDA', 'AMD', meta)
print('NVDA vs AMD')
print(' structural :', result['structural_score'])
print(' description:', result['description_score'])
print(' combined   :', result['combined_score'])
print(' breakdown  :', result['breakdown'])

NVDA vs AMD
 structural : 1.0
 description: 0.1907
 combined   : 0.6763
 breakdown  : {'cross_class': False, 'industry_match': True, 'sector_match': True, 'industry': 'Semiconductors', 'sector': 'Technology'}


In [5]:
df = find_pairs('NVDA', meta, min_combined=0.3)
print(f'{len(df)} candidates for NVDA at min_combined=0.3')
df.head(10)

39 candidates for NVDA at min_combined=0.3


,symbol,asset_class,structural_score,description_score,combined_score,industry,sector,name
0,INTC,equities,1.0,0.2790,0.7116,Semiconductors,Technology,Intel Corporation
1,AVGO,equities,1.0,0.1934,0.6774,Semiconductors,Technology,Broadcom Inc.
2,AMD,equities,1.0,0.1722,0.6689,Semiconductors,Technology,"Advanced Micro Devices, Inc."
3,SITM,equities,1.0,0.1696,0.6678,Semiconductors,Technology,SiTime Corporation
4,MXL,equities,1.0,0.1341,0.6536,Semiconductors,Technology,"MaxLinear, Inc."
5,MU,equities,1.0,0.1187,0.6475,Semiconductors,Technology,"Micron Technology, Inc."
6,AMBQ,equities,1.0,0.1139,0.6455,Semiconductors,Technology,"Ambiq Micro, Inc."
7,QCOM,equities,1.0,0.1099,0.6440,Semiconductors,Technology,QUALCOMM Incorporated
8,QRVO,equities,1.0,0.1052,0.6421,Semiconductors,Technology,"Qorvo, Inc."
9,SMTC,equities,1.0,0.0958,0.6383,Semiconductors,Technology,Semtech Corporation


In [6]:
# Sanity checks
assert not df.empty, 'expected at least some candidates for NVDA'
assert (df['combined_score'] >= 0.3).all()
assert df['combined_score'].is_monotonic_decreasing, 'expected results sorted descending'
assert 'AMD' in df['symbol'].values

# Cross-class comparison (equity vs ETF) — should lean entirely on description score
cross = compare_pair('NVDA', 'XLK', meta)
assert cross['structural_score'] == 0.0, 'cross-class structural score should be 0'
assert cross['combined_score'] == cross['description_score'], 'cross-class combined should equal description score'
print('NVDA vs XLK (cross-class):', cross['structural_score'], cross['description_score'], cross['combined_score'])

print('\npairs_filter checks passed')

NVDA vs XLK (cross-class): 0.0 0.0 0.0

pairs_filter checks passed
